# 第28讲原单元与中文说明
编号从0开始，保留上游历史输出；实际验证另见复现目录。


## 单元 0（code）

导入Image、torch、math、NumPy；不下载模型、不建立网络。Image(url=...)构造展示对象，不等于浏览器已获取图片。


In [18]:
# 本格说明：导入Image、torch、math、NumPy；不下载模型、不建立网络。Image(url=...)构造展示对象，不等于浏览器已获取图片。
# 中文说明：导入Jupyter展示对象；输出url对象不代表网络图片已验证。
from IPython.display import Image
# 中文说明：真实PyTorch提供张量和autograd，后续不是NumPy模拟模型。
import torch
import math
import numpy as np

## 单元 1（markdown）

问题标题：为什么Transformer需要位置信息？要区分不带mask的attention置换等变与自回归mask本身带来的顺序约束。


## why position encoding in Transformer

## 单元 2（markdown）

X[L,D]分别乘Wq/Wk/Wv得Q/K[L,Dk]和V[L,Dv]；只有Q/K特征维必须相同。QK转置得[L,L]，softmax沿key轴归一化，再乘V。无位置无mask时同步置换Q/K/V会同步置换输出，不是输出完全不变；RoPE角度独立于内容，但Q/K及最终注意力依然依赖内容。


- attention mechanism （Transformer 最特色的）
    - $X\in\mathbb R^{\ell\times d}$
    - $W_k\in\mathbb R^{d\times d_k},W_q\in\mathbb R^{d\times d_k},W_v\in\mathbb R^{d\times d_v}$
    - $Q=XW_q\in\mathbb R^{\ell\times d_k}, K=XW_k\in\mathbb R^{\ell\times d_k}, V=XW_v\in\mathbb R^{\ell\times d_v}$

$$
\text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

$$
A_{ij}=\frac{\exp(\frac{q^T_ik_j}{\sqrt{d_k}})}{\sum_{j'}\exp(\frac{q^T_ik_{j'}}{\sqrt{d_k}})}
$$

- $A_{ij}$ （attention weights， $QK^T$: attention scores） 表示的是位置 $i$ 的词（token）与位置 $j$ 的词（token）的注意力权重，
    - 就是如果 $x_i$/$x_j$ 或者 $q_i$/$q_j$($k_i$/$k_j$) 没有编码位置信息，那么的话，这个weight就跟位置无关，显然在 seq modeling 中是有很大缺陷的
    - 也就是一句话的含义，肯定跟 token 的组织有序有关

## 单元 3（markdown）

BERT/GPT示例标题；不能据此概括所有名为GPT的后续模型。


## bert, gpt

## 单元 4（code）

远程示意图URL保留，实际只构造HTML图片对象。没有验证该远程图片内容；本讲用公式与实际小模型embedding解释，不把外链可构造当图已下载。


In [2]:
# 本格说明：远程示意图URL保留，实际只构造HTML图片对象。没有验证该远程图片内容；本讲用公式与实际小模型embedding解释，不把外链可构造当图已下载。
# 中文说明：保留原相对图片路径；公开目录带有对应本地原图，远程URL另标未验证。
Image(url='https://miro.medium.com/v2/resize:fit:786/format:webp/1*iJqlhZz-g6ZQJ53-rE9VvA.png', width=500)

## 单元 5（markdown）

两个Markdown源码片段：BERT加token/type/position embedding；GPT风格把token与position相加，可选token_type项。它们是摘录而非完整模型。代码注释版在块内也增加中文注释；第27讲已实际演示GPT2左padding的位置编号问题。


- BERT: 加性 (absolue) position encoding （learnable position encoding）

    ```python
    # modeling_bert.py
    # 中文说明：绝对位置表查找后与token/type逐元素相加，位置参数是可训练的。
    embeddings = inputs_embeds + token_type_embeddings + self.position_embeddings(position_ids)
    ```

- GPT: 加性 （absolute）position encoding（learnable position encoding）

    ```python
    # modeling_gpt.py
    if inputs_embeds is None:
        # 中文说明：按token ID查token embedding，而不是按位置ID查。
        inputs_embeds = self.wte(input_ids)
    # 中文说明：按绝对position ID查可学习位置表。
    position_embeds = self.wpe(position_ids)
    # 中文说明：同shape向量相加；这里只是摘录，需要其余变量已在真实模型中定义。
    hidden_states = inputs_embeds + position_embeds + token_type_embeds
    ```

## 单元 6（markdown）

固定正弦位置编码标题。绝对位置与是否可学习是两个不同分类维度。


### sin position encoding

## 单元 7（markdown）

偶数通道sin、奇数通道cos，位置t与通道i不能混淆；相邻两通道共享一个频率。d_model=4时频率1与0.01，pos0是[0,1,0,1]。没有可训练位置参数，但仍能通过相位关系表达相对距离。


- 无需训练; 依然是绝对位置编码
- transformers 原始论文

$$
\begin{split}
PE(t,2i)&=\sin(\frac{t}{10000^{\frac{2i}{d_{model}}}})\\
PE(t,2i+1)&=\cos(\frac{t}{10000^{\frac{2i}{d_{model}}}})\\
\Downarrow\\
PE(t,i)&=\sin(\frac{t}{10000^{\frac{i}{d_{model}}}}), \quad \text{i is even}\\
PE(t,i)&=\cos(\frac{t}{10000^{\frac{i-1}{d_{model}}}}), \quad \text{i is odd}\\
\end{split}
$$

d_model = 4

- pos = 0, $[\sin(0),\cos(0), \sin(0),\cos(0)]$
- pos = 1, $[\sin\left(\frac{1}{10000^{0/4}}\right),\cos\left(\frac{1}{10000^{0/4}}\right), \sin\left(\frac{1}{10000^{2/4}}\right), \cos\left(\frac{1}{10000^{2/4}}\right)]$
- pos = 2, $[\sin\left(\frac{2}{10000^{0/4}}\right),\cos\left(\frac{2}{10000^{0/4}}\right), \sin\left(\frac{2}{10000^{2/4}}\right), \cos\left(\frac{2}{10000^{2/4}}\right)]$
- pos = 3, $[\sin\left(\frac{3}{10000^{0/4}}\right),\cos\left(\frac{3}{10000^{0/4}}\right), \sin\left(\frac{3}{10000^{2/4}}\right), \cos\left(\frac{3}{10000^{2/4}}\right)]$




## 单元 8（markdown）

LLaMA RoPE标题；这里频率dim应为head_dim，而非把整个hidden_size当一个head。


## llama RoPE

## 单元 9（markdown）

RoPE将Q/K旋转，以Rm转置Rn=R(n-m)得到带符号相对位移；不是只用绝对值距离。freqs_cis实际是复数相位表，不是显式D×D矩阵。Markdown中的wq/wk/wv/apply/复数变量来自不同层次摘录，不能单独当完整可运行Attention。


- 从绝对位置编码到相对位置编码
    - 绝对位置编码，位置 pos_i 的编码仅取决于 pos_i 的值；
    - 相对位置编码，（一般不需要对每个位置进行单独的编码），而是直接对位置之间的相对距离进行编码
        - pos=0 与 pos=1 的相对位置 $f(|0-1|)$
        - pos=1 与 pos=3 的相对位置 $f(|1-3|)$
        - 偏差构成的矩阵，称为 id 矩阵；
- RoPE
    - 旋转位置编码，为相对位置编码，非加性位置编码，直接嵌入到 attention mechanism 的计算中；
    - $R^d_{\Theta,m}$：位置 $m$ 对应的旋转矩阵 not learnable：非学习的，全局固定的；
        - $m\theta$：frequency


$$
\begin{split}
f(q,m)^Tf(k,n)&=(R_mq)^T(R_nk)\\
&=q^T(R^T_mR_n)k\\
&=q^TR_{n-m}k
\end{split}
$$


```python
# freqs_cis 是一个全局的旋转矩阵
xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)
# 中文说明：对Q/K施加RoPE；V不在这个函数调用里旋转。
xq, xk = apply_rotary_emb(xq, xk, freqs_cis=freqs_cis)
# 中文说明：复数实虚部变回二维实通道，再flatten恢复head维；需有完整前置变量。
xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
# 中文说明：复数实虚部变回二维实通道，再flatten恢复head维；需有完整前置变量。
xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
xq, xk, xv
```

## 单元 10（code）

展示原RoPE论文图，原图已按固定Git字节保留。theta是角频率，m*theta是旋转角，两者要区分。


In [33]:
# 本格说明：展示原RoPE论文图，原图已按固定Git字节保留。theta是角频率，m*theta是旋转角，两者要区分。
# d: dim, 
# m: position
# 中文说明：保留原相对图片路径；公开目录带有对应本地原图，远程URL另标未验证。
Image(url='../../imgs/rope_paper.png', width=600)

## 单元 11（code）

定义完整频率预计算函数。arange隔2取通道，end个位置与D/2频率outer成[end,D/2]角度，polar把单位幅值和角度转为complex64。函数本身没检查dim为偶数，也不返回旋转后的Q/K。


In [4]:
# 本格说明：定义完整频率预计算函数。arange隔2取通道，end个位置与D/2频率outer成[end,D/2]角度，polar把单位幅值和角度转为complex64。函数本身没检查dim为偶数，也不返回旋转后的Q/K。
# 中文说明：参数dim=head维度、end=位置个数、theta=基础常数；输出complex64[end,dim/2]。
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    # 中文说明：每对通道一个频率，指数是2r/dim；不是位置m本身。
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    # 中文说明：产生绝对位置0到end-1，右端不含end。
    t = torch.arange(end)  # type: ignore
    # 中文说明：位置[end]外积频率[D/2]，广播关系得到所有旋转角。
    freqs = torch.outer(t, freqs).float()  # type: ignore
    # 中文说明：幅值和弧度角转复数；幅值为1时得到cos+i sin。
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # complex64
    return freqs_cis

## 单元 12（code）

调用D=4/end=3，得到complex64[3,2]；两条频率1和0.01。只生成相位表，不涉及模型权重或训练。


In [13]:
# 本格说明：调用D=4/end=3，得到complex64[3,2]；两条频率1和0.01。只生成相位表，不涉及模型权重或训练。
cis = precompute_freqs_cis(dim=4, end=3)
cis

tensor([[ 1.0000+0.0000j,  1.0000+0.0000j],
        [ 0.5403+0.8415j,  0.9999+0.0100j],
        [-0.4161+0.9093j,  0.9998+0.0200j]])

## 单元 13（markdown）

对上一格每步手算。符号j为虚数单位，此处不是key下标；e^(j angle)=cos(angle)+j sin(angle)。


$$
\begin{split}
&\text{freqs}=[1,\frac{1}{\theta^{2/4}}]=[1., 0.01]\\
&t=[0,1,2]\\
&\text{freqs}=\begin{bmatrix}
0 & 0\\
1 & 0.01\\
2 & 0.02
\end{bmatrix}\\
&\text{freqs\_cis}=e^{j\cdot\text{freqs}}=\begin{bmatrix}
1 & 1\\
e^j & e^{j\cdot0.01}\\
e^{j\cdot2} & e^{j\cdot0.02}
\end{bmatrix}
\end{split}
$$

## 单元 14（code）

polar([1],[2])把幅值1和弧度2组成单位复数，约-0.4161+0.9093j；不是e的实指数。


In [21]:
# 本格说明：polar([1],[2])把幅值1和弧度2组成单位复数，约-0.4161+0.9093j；不是e的实指数。
# torch.polar(torch.tensor([1.]), torch.tensor([0.]))
# 中文说明：幅值和弧度角转复数；幅值为1时得到cos+i sin。
torch.polar(torch.tensor([1.]), torch.tensor([2.]))

tensor([-0.4161+0.9093j])

## 单元 15（code）

手写[3,2]角度矩阵、ones_like幅值和polar，验证与预计算相位表相同。ones_like继承shape/dtype，不是创建训练Parameter。


In [10]:
# 本格说明：手写[3,2]角度矩阵、ones_like幅值和polar，验证与预计算相位表相同。ones_like继承shape/dtype，不是创建训练Parameter。
theta_matrix = torch.tensor([
    [0.0, 0.0],
    [1.0, 0.01],
    [2.0, 0.02]
])

# 幅度矩阵，全为 1
# 中文说明：保持角度矩阵的shape/dtype，单位圆幅值均为1。
r_matrix = torch.ones_like(theta_matrix)

# 计算 e^{j*theta_matrix}
# 中文说明：幅值和弧度角转复数；幅值为1时得到cos+i sin。
e_j_theta_matrix = torch.polar(r_matrix, theta_matrix)
e_j_theta_matrix

tensor([[ 1.0000+0.0000j,  1.0000+0.0000j],
        [ 0.5403+0.8415j,  0.9999+0.0100j],
        [-0.4161+0.9093j,  0.9998+0.0200j]])

## 单元 16（markdown）

苏剑林RoPE推导链接，属于作者解释材料；本讲保留来源，不把外链当命令。


### https://spaces.ac.cn/archives/8265

## 单元 17（markdown）

R12/R34若是完整D×D嵌入块可相乘；若指两个2×2小块，应写block_diag而不是小矩阵连乘。本例20格使用完整4×4矩阵，所以相乘合法且作用在不相交维度。


- 二维向量的旋转矩阵是 $2\times 2$的正交矩阵
- d维向量的旋转矩阵是 d*d，依然是正交矩阵
    - 注意 $x_1,x_2$ 对应一个2*2旋转矩阵 $R_{1,2}$
    - $x_3,x_4$ 对应一个2*2的旋转矩阵 $R_{3,4}$
    - ...
- $R=R_{1,2}R_{3,4}R_{5,6}\cdots R_{d-1,d}$

## 单元 18（code）

展示四维空间六种坐标平面的旋转；后面20格只选择互不相交的(1,2)与(3,4)两个平面，不是把图中六个旋转全部相乘。


In [34]:
# 本格说明：展示四维空间六种坐标平面的旋转；后面20格只选择互不相交的(1,2)与(3,4)两个平面，不是把图中六个旋转全部相乘。
# 中文说明：保留原相对图片路径；公开目录带有对应本地原图，远程URL另标未验证。
Image(url='../../imgs/4d_rotation.png', width=300)

## 单元 19（code）

展示相邻维配对图，向量1/2、3/4分别旋转。不要与23格的前后半区配对混用。


In [3]:
# 本格说明：展示相邻维配对图，向量1/2、3/4分别旋转。不要与23格的前后半区配对混用。
# 中文说明：保留原相对图片路径；公开目录带有对应本地原图，远程URL另标未验证。
Image(url='../../imgs/rope_1.png', width=450)

## 单元 20（code）

构造两个4×4嵌入旋转，角pi/6与pi/3，torch.mm相乘得R。torch.tensor([...tensor表达式...])会拷贝数值并切断这些角度的原计算图；适合展示，不是可微构造范例。


In [24]:
# 本格说明：构造两个4×4嵌入旋转，角pi/6与pi/3，torch.mm相乘得R。torch.tensor([...tensor表达式...])会拷贝数值并切断这些角度的原计算图；适合展示，不是可微构造范例。
theta1 = torch.tensor(np.pi/6)
theta2 = torch.tensor(np.pi/3)

R12 = torch.tensor([
    [torch.cos(theta1), -torch.sin(theta1), 0, 0],
    [torch.sin(theta1),  torch.cos(theta1), 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
])

R34 = torch.tensor([
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, torch.cos(theta2), -torch.sin(theta2)],
        [0, 0, torch.sin(theta2),  torch.cos(theta2)]
])

# 中文说明：二维矩阵乘法；不要用逐元素乘法代替旋转复合。
R = torch.mm(R12, R34)

R12, R34, R

(tensor([[ 0.8660, -0.5000,  0.0000,  0.0000],
         [ 0.5000,  0.8660,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  1.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  1.0000]]),
 tensor([[ 1.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  1.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.5000, -0.8660],
         [ 0.0000,  0.0000,  0.8660,  0.5000]]),
 tensor([[ 0.8660, -0.5000,  0.0000,  0.0000],
         [ 0.5000,  0.8660,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.5000, -0.8660],
         [ 0.0000,  0.0000,  0.8660,  0.5000]]))

## 单元 21（code）

第一行仅检验RR转置与R转置R相同，正规矩阵也满足，单独不足证明正交；第二行与单位阵比较才给出当前例子的正交数值检查。


In [30]:
# 本格说明：第一行仅检验RR转置与R转置R相同，正规矩阵也满足，单独不足证明正交；第二行与单位阵比较才给出当前例子的正交数值检查。
# 中文说明：二维矩阵乘法；不要用逐元素乘法代替旋转复合。
print(torch.allclose(torch.mm(R, R.T), torch.mm(R.T, R)))
# 中文说明：二维矩阵乘法；不要用逐元素乘法代替旋转复合。
print(torch.allclose(torch.mm(R, R.T), torch.eye(4)))

True
True


## 单元 22（code）

图中仍是相邻配对的逐元素cos/sin实现；接下来23格函数改用前后半区配对，必须额外说明维度置换，不能当作同一坐标顺序的直接延续。


In [4]:
# 本格说明：图中仍是相邻配对的逐元素cos/sin实现；接下来23格函数改用前后半区配对，必须额外说明维度置换，不能当作同一坐标顺序的直接延续。
# 中文说明：保留原相对图片路径；公开目录带有对应本地原图，远程URL另标未验证。
Image(url='../../imgs/rope_2.png', width=450)

## 单元 23（code）

完整get_rope_matrix：cos/sin各[D/2]，填入四个半区对角块，配对坐标(r,r+D/2)。torch.zeros默认CPU float32，因此即使theta是double，也会写入float32矩阵。维度奇数/长度不匹配缺少显式校验。


In [10]:
# 本格说明：完整get_rope_matrix：cos/sin各[D/2]，填入四个半区对角块，配对坐标(r,r+D/2)。torch.zeros默认CPU float32，因此即使theta是double，也会写入float32矩阵。维度奇数/长度不匹配缺少显式校验。
# theta: m*theta
# m: position
# 中文说明：参数d为偶数维，theta需有d/2个角；返回半区配对矩阵，不是相邻配对布局。
def get_rope_matrix(d, theta):
    """Construct the ROPE rotation matrix."""
    cos_theta = torch.cos(theta)
    sin_theta = torch.sin(theta)
    # 中文说明：默认CPU float32；不会自动跟随theta的dtype/device。
    mat = torch.zeros(d, d)
    # 中文说明：左上对角块cos，各半区对应同一频率。
    mat[0:d//2, 0:d//2] = torch.diag(cos_theta)
    # 中文说明：右下对角块cos；与左上角共享角度。
    mat[d//2:, d//2:] = torch.diag(cos_theta)
    # 中文说明：右上角-sin，对应二维旋转的负交叉项。
    mat[0:d//2, d//2:] = -torch.diag(sin_theta)
    # 中文说明：左下角+sin；与右上共同将两半区配成旋转平面。
    mat[d//2:, 0:d//2] = torch.diag(sin_theta)
    return mat

## 单元 24（code）

d=4；列表推导逐一把角度换成弧度。theta_m/n各[2]，只是展示两组角，不是训练得到的位置向量。若要推广到m/n任意位置，应保持同一组基础频率的线性关系。


In [5]:
# 本格说明：d=4；列表推导逐一把角度换成弧度。theta_m/n各[2]，只是展示两组角，不是训练得到的位置向量。若要推广到m/n任意位置，应保持同一组基础频率的线性关系。
# 示例向量的维度
d = 4  # 必须是偶数
theta_m = torch.tensor([i * math.pi / 180 for i in range(d//2)])
theta_n = torch.tensor([(i + 1) * math.pi / 180 for i in range(d//2)])

## 单元 25（code）

Jupyter展示两组角度tensor的tuple，不做矩阵乘法。


In [7]:
# 本格说明：Jupyter展示两组角度tensor的tuple，不做矩阵乘法。
theta_m, theta_n

(tensor([0.0000, 0.0175]), tensor([0.0175, 0.0349]))

## 单元 26（code）

逐元素角差，各维约一度；相对旋转保留差的正负。


In [8]:
# 本格说明：逐元素角差，各维约一度；相对旋转保留差的正负。
theta_n - theta_m

tensor([0.0175, 0.0175])

## 单元 27（code）

构造同值两元素tensor核对角差；列表推导中的i未使用，重复常数并非两个不同频率。


In [9]:
# 本格说明：构造同值两元素tensor核对角差；列表推导中的i未使用，重复常数并非两个不同频率。
torch.tensor([1 * math.pi / 180 for i in range(d//2)])

tensor([0.0175, 0.0175])

## 单元 28（code）

用同一半区配对函数建立Rm/Rn，shape各[4,4]。


In [38]:
# 本格说明：用同一半区配对函数建立Rm/Rn，shape各[4,4]。
# 构造 R_m 和 R_n
R_m = get_rope_matrix(d, theta_m)
R_n = get_rope_matrix(d, theta_n)

## 单元 29（code）

Rm.T是转置view；用角差建立相对矩阵R(n-m)。这里矩阵T不是论文最大位置数T。


In [42]:
# 本格说明：Rm.T是转置view；用角差建立相对矩阵R(n-m)。这里矩阵T不是论文最大位置数T。
R_m_T = R_m.T
R_n_minus_m = get_rope_matrix(d, theta_n - theta_m)
R_n_minus_m

tensor([[ 0.9998,  0.0000, -0.0175, -0.0000],
        [ 0.0000,  0.9998, -0.0000, -0.0175],
        [ 0.0175,  0.0000,  0.9998,  0.0000],
        [ 0.0000,  0.0175,  0.0000,  0.9998]])

## 单元 30（code）

实际计算Rm转置乘Rn；应与上一格R(n-m)数值接近，不能只看打印的矩阵就认为所有维度已经证明。


In [41]:
# 本格说明：实际计算Rm转置乘Rn；应与上一格R(n-m)数值接近，不能只看打印的矩阵就认为所有维度已经证明。
# 中文说明：二维矩阵乘法；不要用逐元素乘法代替旋转复合。
product = torch.mm(R_m_T, R_n)
product

tensor([[ 0.9998,  0.0000, -0.0175,  0.0000],
        [ 0.0000,  0.9998,  0.0000, -0.0175],
        [ 0.0175,  0.0000,  0.9998,  0.0000],
        [ 0.0000,  0.0175,  0.0000,  0.9998]])

## 单元 31（code）

用q/k两个[4]向量计算旋转结果；原result_1那行被注释，所以原notebook没有真正执行第一条dot比较。独立验证中另行计算三种表达式。


In [43]:
# 本格说明：用q/k两个[4]向量计算旋转结果；原result_1那行被注释，所以原notebook没有真正执行第一条dot比较。独立验证中另行计算三种表达式。
# 测试向量 q 和 k
q = torch.tensor([1.0, 2.0, 3.0, 4.0])
k = torch.tensor([4.0, 3.0, 2.0, 1.0])

# 计算 (R_m q)^T (R_n k)
# 中文说明：二维矩阵乘一维向量，输出仍为一维向量。
R_m_q = torch.mv(R_m, q)
# 中文说明：二维矩阵乘一维向量，输出仍为一维向量。
R_n_k = torch.mv(R_n, k)
# result_1 = torch.dot(R_m_q, R_n_k)

## 单元 32（code）

计算q转置Rm转置Rn k，返回零维浮点tensor。


In [46]:
# 本格说明：计算q转置Rm转置Rn k，返回零维浮点tensor。
# 计算 q^T R_m^T R_n k
# 中文说明：二维矩阵乘一维向量，输出仍为一维向量。
result_2 = torch.dot(q, torch.mv(product, k))
result_2

tensor(20.3460)

## 单元 33（code）

计算q转置R(n-m) k；与上格比较可核对该例相对旋转。新增验证还检查反向梯度与公共位置平移。


In [47]:
# 本格说明：计算q转置R(n-m) k；与上格比较可核对该例相对旋转。新增验证还检查反向梯度与公共位置平移。
# 计算 q^T R_{n-m} k
# 中文说明：二维矩阵乘一维向量，输出仍为一维向量。
result_3 = torch.dot(q, torch.mv(R_n_minus_m, k))
result_3

tensor(20.3460)

## 单元 34（markdown）

切换到CoPE论文标题；它是2024 Contextual Position Encoding，不是同名Clipped RoPE/Complex PE。


## CoPE：Contextual Position Encoding: Learning to Count What’s Important

## 单元 35（markdown）

论文和第三方链接保留。CoPE根据query/key门控来计数，可能学到句子/词等单位；这不是保证未经训练就能识别句子。普通相对PE并非在所有任务都无法定位句子，论文的简化假设和实验需分开看。


- https://arxiv.org/pdf/2405.18719
- https://www.zhihu.com/question/657761483/answer/3517582623
- 相对位置编码
    - 之前的哪怕是 rope，都是基于 token positions 的，独立于上下文
    - cope （$p_{ij}$ 的计算已经考虑了 $q_i,k_j$）不只可以 attend 到 token，还可以到 sentence 到 paragraph

$$
a_{ij}=\text{Softmax}(\mathbf q^T_i(\mathbf k_j+\mathbf e[i-j]))
$$

## 单元 36（code）

论文Figure1原图：门控句号形成同句共享的理想位置；图中0/1门是说明用，有限logit经sigmoid通常是0到1之间小数，不是硬分类。


In [15]:
# 本格说明：论文Figure1原图：门控句号形成同句共享的理想位置；图中0/1门是说明用，有限logit经sigmoid通常是0到1之间小数，不是硬分类。
# rel p_{ij} 显然是整数
# cope, p_{ij} 显然不是整数
# 这个例子想说的是我们要关注最后一句话， relative pe 并不能做到很好的 attend 到最后一句话，cope 可以
# 中文说明：保留原相对图片路径；公开目录带有对应本地原图，远程URL另标未验证。
Image(url='../../imgs/cope.png', width=500)

## 单元 37（code）

原相对位置矩阵示意图；其i-j与论文CoPE含当前项的求和i-j+1有一个偏移约定。


In [12]:
# 本格说明：原相对位置矩阵示意图；其i-j与论文CoPE含当前项的求和i-j+1有一个偏移约定。
# 中文说明：保留原相对图片路径；公开目录带有对应本地原图，远程URL另标未验证。
Image(url='../../imgs/rel_position_matrix.png', width=400)

## 单元 38（markdown）

公式链：sigmoid gate[B,Q,K]、沿key从右往左累加为context position、在整数embedding相邻槽间插值，再加到attention logit。先算query乘位置表，避免物化[B,Q,K,D]。因果mask必须在算gate前进入logit，防止未来key参与计数。


- $g_{ij}=\sigma(q_i^Tk_j)$（gate value）, $i$ 称为 target，$j\lt i$, 也就是 $i$ 左边的；
    - 可以想象自回归的过程，$i$表示当前位置，$j$则表示过去的已经遍历/生成的 tokens；
- $p_{ij}=\sum_{k=j}^ig_{ik}$
    - 如果 $g_{ij}==1$, $p_{ij}=i-j+1$
    - 显然 $p_{ij}$ 大概率不是整数，它是我们希望刻画的相对位置；
- a learnable embedding vector $\mathbf e[p]$（$p\in[0,T]$），做插值；

    $$
    \mathbf{e}[p_{ij}] = (p_{ij} - \lfloor p_{ij} \rfloor) \mathbf{e} \left[ \lceil p_{ij} \rceil \right] + (1 - p_{ij} + \lfloor p_{ij} \rfloor) \mathbf{e} \left[ \lfloor p_{ij} \rfloor \right].
    $$

- attention weights

    $$
    a_{ij}=\text{Softmax}(\mathbf q_i^T(\mathbf k_j+\mathbf e[p_{ij}]))
    $$
  - computing and storing vectors $\mathbf e[p_{ij}]$ uses extra compute and memory.

$$
\begin{align}
z_i[p] &= \mathbf{q}_i^\top \mathbf{e}[p] \quad \text{for } p \in \{0, 1, \ldots, T\} \\
z_i[p_{ij}] &= (p_{ij} - \lfloor p_{ij} \rfloor) z_i \left[ \lceil p_{ij} \rceil \right] + (1 - p_{ij} + \lfloor p_{ij} \rfloor) z_i \left[ \lfloor p_{ij} \rfloor \right] \\
a_{ij} &= \text{Softmax}(\mathbf{q}_i^\top \mathbf{k}_j + z_i[p_{ij}]).
\end{align}
$$

## 单元 39（markdown）

这格是Markdown，但含完整CoPE和SelfAttn两个nn.Module类，必须阅读和实际核对。提取到独立Python并导入真实torch.nn即可运行。CoPE返回位置logit偏置[B,Q,K]，SelfAttn返回对V加权后的[B,Q,Dv]；这里没有QKV投影、head拆分、残差或完整语言模型。Parameter位置表零初始化，第一步位置bias为0，需训练后才产生非零位置作用。


```python
# 中文说明：nn.Module子类，负责位置偏置；类定义本身还没有建立实例。
class CoPE(nn.Module):
    def __init__(self, npos_max, head_dim):
        # 中文说明：先初始化Module，之后赋给属性的Parameter/子Module才能被正常登记。
        super().__init__()
        # 中文说明：保存整数位置表槽数P；合法索引0到P-1，不等于最大token长度。
        self.npos_max = npos_max
        # 中文说明：注册可训练表[1,D,P]，初始全0；不是nn.Embedding对象但同样由optimizer发现。
        self.pos_emb = nn.parameter.Parameter(torch.zeros(1, head_dim, npos_max))

    # 中文说明：query[B,Q,D]和已缩放且已mask的attn_logits[B,Q,K]，返回位置bias同[B,Q,K]。
    def forward(self, query, attn_logits):
        # compute positions
        # 中文说明：sigmoid把每个内容分数映到(0,1)；屏蔽位置logit=-inf时门值为0。
        gates = torch.sigmoid(attn_logits)
        # 中文说明：右到左累积：pos[...,j]=sum_{k=j}^{K-1}gate[...,k]；因果mask使未来贡献0。
        pos = gates.flip(-1).cumsum(dim=-1).flip(-1)
        # 中文说明：上界P-1防止gather越界；超过上界的位置都落到末槽，距离区分会饱和。
        pos = pos.clamp(max=self.npos_max - 1)
        # interpolate from integer positions
        # 中文说明：向上取整再转int64索引；floor/ceil索引本身不可微。
        pos_ceil = pos.ceil().long()
        # 中文说明：向下取整槽；小数权重仍从浮点pos保留梯度。
        pos_floor = pos.floor().long()
        # 中文说明：[B,Q,D]乘[1,D,P]广播成[B,Q,P]；先点积再插值避免四维位置向量。
        logits_int = torch.matmul(query, self.pos_emb)
        # 中文说明：按[B,Q,K]索引取右槽分数，输出[B,Q,K]。
        logits_ceil = logits_int.gather(-1, pos_ceil)
        # 中文说明：取左槽分数；两个gather共享query和位置表的梯度。
        logits_floor = logits_int.gather(-1, pos_floor)
        # 中文说明：小数部分在[0,1)，pos恰为整数时两个索引相等，此代码选择的门梯度可能为0。
        w = pos - pos_floor
        # 中文说明：线性插值，前向等价于先插值embedding再与query点积。
        return logits_ceil * w + logits_floor * (1 - w)

# 中文说明：单head注意力示例，Q/K/V由调用者传入，不在类中训练投影。
class SelfAttn(nn.Module):
    def __init__(self, npos_max, head_dim):
        # 中文说明：先初始化Module，之后赋给属性的Parameter/子Module才能被正常登记。
        super().__init__()
        # 中文说明：注册CoPE子模块，SelfAttn.parameters会包含其位置表。
        self.cope = CoPE(npos_max, head_dim)
        # 中文说明：保存Q/K的最后维度D，用sqrt(D)缩放内容分数。
        self.head_dim = head_dim

    # 中文说明：query/key/val为三维batch张量；mask须是允许=1/屏蔽=0的可广播张量，不是加性0/-inf mask。
    def forward(self, query, key, val, mask):
        # q, k, v have dimensions batch x seq_len x head_dim
        # 中文说明：[B,Q,D]乘[B,D,K]得[B,Q,K]，bmm不自动广播batch。
        attn_logits = torch.bmm(query, key.transpose(-1, -2))
        # 中文说明：缩放内容分数；CoPE门直接使用这份已缩放分数。
        attn_logits = attn_logits / math.sqrt(self.head_dim)
        # 中文说明：二值mask的log变为0/-inf，先屏蔽再计数；全屏蔽行softmax会NaN。
        attn_logits += mask.log()
        # 中文说明：将位置偏置加回同一logit；输出不是把position加到hidden states。
        attn_logits += self.cope(query, attn_logits)
        # 中文说明：沿最后key维归一化；每条非空有效行权重和为1。
        attn = torch.softmax(attn_logits, dim=-1)
        # 中文说明：[B,Q,K]乘V[B,K,Dv]返回[B,Q,Dv]，没有额外output投影。
        out = torch.bmm(attn, val)
        return out
```